In [ ]:
import subprocess, sys

subprocess.run([
    sys.executable, "-m", "pip", "install",
    "transformers>=4.41.0",
    "datasets",
    "nltk",
    "--quiet"
], check=True)

# Libraries

In [1]:
import os, random, time, json
import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    GPT2LMHeadModel,
    GPT2Tokenizer,
    get_linear_schedule_with_warmup   
)
from torch.optim import AdamW

torch.backends.cudnn.benchmark =True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"  GPU      : {gpu.name}")
    print(f"  VRAM     : {gpu.total_memory / 1e9:.1f} GB")

In [ ]:
ROCSTORIES_PATH = "/kaggle/input/rocstories-and-story-cloze-test-corpora/rocstories_spring2016.csv""

CONFIG = {
    "model_name"     : "gpt2-medium",

    "batch_size"     : 8,
    "epochs"         : 5,
    "lr"             : 3e-5,
    "warmup_steps"   : 100,
    "max_grad_norm"  : 1.0,

    "max_length"     : 256,      
    "train_split"    : 0.9,      
    "max_samples"    : None,

    # Paths
    "save_dir"       : "/kaggle/working/gpt2_story",

    # Misc
    "seed"           : 42,
    "log_every"      : 50,
    "device"         : DEVICE
}
os.makedirs(CONFIG["save_dir"], exist_ok=True)

torch.manual_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
random.seed(CONFIG["seed"])

Load & Explore ROCStories Dataset

In [ ]:
df = pd.read_csv(ROCSTORIES_PATH)
print(f"Total stories : {len(df):,}")
print(f"Columns       : {list(df.columns)}")
print(f"\nSample story:")
print(f"  Title : {df.iloc[0]['storytitle']}")
print(f"  S1    : {df.iloc[0]['sentence1']}")
print(f"  S2    : {df.iloc[0]['sentence2']}")
print(f"  S3    : {df.iloc[0]['sentence3']}")
print(f"  S4    : {df.iloc[0]['sentence4']}")
print(f"  S5    : {df.iloc[0]['sentence5']}")


In [ ]:
def build_story_text(row):
    story = (
        f"<|story|> "
        f"{row['storytitle']}. "
        f"{row['sentence1']} "
        f"{row['sentence2']} "
        f"{row['sentence3']} "
        f"{row['sentence4']} "
        f"{row['sentence5']} "
        f"<|endofstory|>"
    )
    return story

df["full_story"] = df.apply(build_story_text, axis=1)

if CONFIG["max_samples"]:
    df = df.sample(CONFIG["max_samples"], random_state=CONFIG["seed"])


split_idx  = int(len(df) * CONFIG["train_split"])
train_df   = df.iloc[:split_idx].reset_index(drop=True)
val_df     = df.iloc[split_idx:].reset_index(drop=True)

print(f"Train stories : {len(train_df):,}")
print(f"Val stories   : {len(val_df):,}")
print(f"\nExample formatted story:")
print(f"  {df['full_story'].iloc[0]}")